In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import joblib
import ta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [2]:
# Configurações Base
time_steps = 7
tickers = {
    'Bovespa': '^BVSP',        
    'Dolar': 'BRL=X',        
    'SP500': '^GSPC',        
    'Shanghai': '000001.SS', # Bolsa da China
    'Petroleo': 'BZ=F',      
    'Minerio': 'TIO=F',      
    'Ouro': 'GC=F'    
}
nomes_classes = {0: "📉 BAIXA", 1: "➖ NEUTRO", 2: "📈 ALTA"}

In [3]:
print("📥 Lendo o fechamento do mercado global...")
df_precos = yf.download(list(tickers.values()), period='2mo')['Close']
df_precos.rename(columns={v: k for k, v in tickers.items()}, inplace=True)
df_precos.ffill(inplace=True)
df_precos.dropna(inplace=True)

📥 Lendo o fechamento do mercado global...


[*********************100%***********************]  7 of 7 completed


In [4]:
df_features = pd.DataFrame(index=df_precos.index)
for col in df_precos.columns:
    df_features[col] = df_precos[col].pct_change()

In [5]:
# Injetando a Análise Técnica (O segredo do modelo)
df_features['RSI'] = ta.momentum.RSIIndicator(df_precos['Bovespa'], window=14).rsi()
sma_15 = ta.trend.SMAIndicator(df_precos['Bovespa'], window=15).sma_indicator()
df_features['Distancia_SMA15'] = (df_precos['Bovespa'] / sma_15) - 1
df_features['Bollinger_Width'] = ta.volatility.BollingerBands(df_precos['Bovespa'], window=20).bollinger_wband()

In [6]:
# Limpar buracos e isolar os últimos 7 dias exatos
df_features.dropna(inplace=True)
colunas_features = df_features.columns
ultimos_dias = df_features.tail(time_steps).values

print(f"✅ Dados processados com sucesso! (Base de {df_precos.index[-time_steps].strftime('%d/%m')} até {df_precos.index[-1].strftime('%d/%m')})")

✅ Dados processados com sucesso! (Base de 12/05 até 20/05)


In [7]:
caminho_modelos = os.path.join(os.getcwd(), 'modelos')

In [8]:
# 3.1 Carrega a Régua Matemática (Scaler)
scaler_X = joblib.load(os.path.join(caminho_modelos, 'scaler_X_classificador_3c.pkl'))
X_futuro_scaled = scaler_X.transform(ultimos_dias.reshape(-1, ultimos_dias.shape[-1])).reshape(1, time_steps, len(colunas_features))

In [9]:
# 3.2 Reconstrói a Arquitetura da Rede e carrega a Memória (Pesos)
modelo_gru = Sequential([
    GRU(50, return_sequences=True, input_shape=(time_steps, len(colunas_features))),
    Dropout(0.2),
    GRU(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(3, activation='softmax')
])
modelo_gru.load_weights(os.path.join(caminho_modelos, 'modelo_gru_classificador_3c.weights.h5'))

C:\Users\flavi\anaconda3\envs\SeriesTemporais\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
probabilidades = modelo_gru.predict(X_futuro_scaled, verbose=0)[0]
classe_vencedora = np.argmax(probabilidades)

In [11]:
fechamento_hoje = df_precos['Bovespa'].iloc[-1]
lim_inf = fechamento_hoje * (1 - 0.002)
lim_sup = fechamento_hoje * (1 + 0.002)

print("\n" + "="*45)
print(" 🚀 RADAR BOVESPA QUANTITATIVO 🚀")
print("="*45)
print(f"Cotação Atual Base: {fechamento_hoje:.0f} pts")
print(f"Veredito do Modelo: {nomes_classes[classe_vencedora]}")

print("\n📊 Raio-X de Probabilidades:")
print(f"   📈 Alta:   {probabilidades[2]*100:>5.2f}%")
print(f"   ➖ Neutro: {probabilidades[1]*100:>5.2f}%")
print(f"   📉 Baixa:  {probabilidades[0]*100:>5.2f}%")

print("\n🎯 ALVOS NO GRÁFICO (PRÓXIMO DIA ÚTIL):")
print(f"   🟢 Confirma ALTA se romper:   {lim_sup:.0f} pts")
print(f"   🟡 ZONA DE INDECISÃO:         {lim_inf:.0f} a {lim_sup:.0f} pts")
print(f"   🔴 Confirma BAIXA se perder:  {lim_inf:.0f} pts")
print("="*45)


 🚀 RADAR BOVESPA QUANTITATIVO 🚀
Cotação Atual Base: 177356 pts
Veredito do Modelo: 📉 BAIXA

📊 Raio-X de Probabilidades:
   📈 Alta:    9.35%
   ➖ Neutro: 12.60%
   📉 Baixa:  78.06%

🎯 ALVOS NO GRÁFICO (PRÓXIMO DIA ÚTIL):
   🟢 Confirma ALTA se romper:   177710 pts
   🟡 ZONA DE INDECISÃO:         177001 a 177710 pts
   🔴 Confirma BAIXA se perder:  177001 pts


In [12]:
ultimos_dias = df_features.tail(time_steps).values
X_futuro_scaled = scaler_X.transform(ultimos_dias.reshape(-1, ultimos_dias.shape[-1])).reshape(1, time_steps, len(colunas_features))

# Probabilidades da Rede Neural
probabilidades = modelo_gru.predict(X_futuro_scaled, verbose=0)[0]
classe_vencedora = np.argmax(probabilidades)

# Matemática para transformar as classes em Pontos Reais
fechamento_hoje = df_precos['Bovespa'].iloc[-1]
limite_inferior = fechamento_hoje * (1 - 0.002) # -0.2%
limite_superior = fechamento_hoje * (1 + 0.002) # +0.2%

print("\n🚀 PREVISÃO PARA O PRÓXIMO DIA ÚTIL 🚀")
print(f"Cotação Base (Fechamento Hoje): {fechamento_hoje:.2f} pts")
print(f"Cenário mais provável: {nomes_classes[classe_vencedora]}")

print("\n📊 Raio-X de Probabilidades:")
for i in range(3):
    print(f"{nomes_classes[i]}: {probabilidades[i]*100:.2f}%")

print("\n🎯 O QUE ISSO SIGNIFICA EM PONTOS PARA AMANHÃ:")
print(f"🟢 Para ser ALTA: Precisa fechar ACIMA de {limite_superior:.0f} pts")
print(f"🔴 Para ser BAIXA: Precisa fechar ABAIXO de {limite_inferior:.0f} pts")
print(f"🟡 Zona NEUTRA: Ficar preso entre {limite_inferior:.0f} e {limite_superior:.0f} pts")


🚀 PREVISÃO PARA O PRÓXIMO DIA ÚTIL 🚀
Cotação Base (Fechamento Hoje): 177355.73 pts
Cenário mais provável: 📉 BAIXA

📊 Raio-X de Probabilidades:
📉 BAIXA: 78.06%
➖ NEUTRO: 12.60%
📈 ALTA: 9.35%

🎯 O QUE ISSO SIGNIFICA EM PONTOS PARA AMANHÃ:
🟢 Para ser ALTA: Precisa fechar ACIMA de 177710 pts
🔴 Para ser BAIXA: Precisa fechar ABAIXO de 177001 pts
🟡 Zona NEUTRA: Ficar preso entre 177001 e 177710 pts
